# Foundation - Lab 4
___

In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import json 
import requests
from pypdf import PdfReader
import gradio as gr
from IPython.display import Markdown, display

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PUSHOVER_USER = os.getenv("PUSHOVER_USER")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN")
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"

load_dotenv(override=True)
openai = OpenAI(
    api_key = OPENAI_API_KEY
)
NAME = "Stefanus Yudi Irwan"
OPENAI_MODEL = "gpt-4.1-nano"

In [3]:
def push(message: str) -> None:
    """
       Function to push notification 
       to pushover
    """
    print(f"Push: {message}")
    payload = {"user": PUSHOVER_USER, "token": PUSHOVER_TOKEN, "message": message}
    requests.post(PUSHOVER_URL, data=payload)

# Test pushover
push("HEY Stefanus Yudi Irwan!")

Push: HEY Stefanus Yudi Irwan!


In [22]:
def record_user_details(email: str,
                        name: str = "Name not provided",
                        notes: str = "not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            },
            "notes": {
                "type": "string",
                "description": "Any additional information about this conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

def record_unknown_question(question: str):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}
    
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

def handle_tool_calls(tool_calls): 
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [23]:
import json
from types import SimpleNamespace

fake_record_user_details = SimpleNamespace(
    id = "record_user_details_001",
    function = SimpleNamespace(
        name = "record_user_details",
        arguments = json.dumps({
            "email": "habieb@riziek.com",
            "name": "Habieb Riziek",
            "notes": "Who are you?"
        })
    )
)

fake_record_unknown_question = SimpleNamespace(
    id = "record_unknown_question_001",
    function = SimpleNamespace(
        name = "record_unknown_question",
        arguments=json.dumps({
            "question": f"What is the internal architecture of GPT-6?"
        })
    )
)

In [24]:
tool_call_results = handle_tool_calls([
    fake_record_user_details,
    fake_record_unknown_question
])

Tool called: record_user_details
Push: Recording interest from Habieb Riziek with email habieb@riziek.com and notes Who are you?
Tool called: record_unknown_question
Push: Recording What is the internal architecture of GPT-6? asked that I couldn't answer


In [25]:
tool_call_results

[{'role': 'tool',
  'content': '{"recorded": "ok"}',
  'tool_call_id': 'record_user_details_001'},
 {'role': 'tool',
  'content': '{"recorded": "ok"}',
  'tool_call_id': 'record_unknown_question_001'}]

In [26]:
reader = PdfReader("data/Resume - Stefanus Yudi Irwan.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text
print(resume)

STEFANUS YUDI IRWAN 
West Java, Indonesia 40526 • +62821-5812-2742 • yudi.stefanus22@gmail.com • personal website 
  
PROFILE 
A results -driven engineer with a unique background in oil and gas field operations and AI product 
development. Driven by a mission to bridge technological gaps in Indonesia's digital infrastructure, I aim to 
specialize in applied and efficient AI systems. I am seeking to leverage my holistic perspective to contribute to 
global technological innovation and develop deployable solutions for real -world engineering challenges. 
 
EDUCATION 
Universitas Gadjah Mada            
Bachelor of Engineering, Engineering Physics 
Cumulative GPA: 3.82/4.00 (Cum Laude)  
• Honors: Ranked #1 in the Engineering Physics graduating class of August 2019  
• Relevant Coursework: Computer Programming, Linear Algebra, Probability & Statistics, Numerical 
Methods, Database System, Automatic Control, Sensor & Actuator, Electronics, Signal Processing 
• Thesis: Development of Coal M

In [27]:
system_prompt = f"You are acting as {NAME}. You are answering questions on {NAME}'s resume, \
    particularly questions related to {NAME}'s career, background, skills and experience. \
    Your responsibility is to represent {NAME} for interactions on the website as faithfully as possible. \
    You are given a summary of {NAME}'s resume which you can use to answer questions. \
    Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
    If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer \
    even if it's about something trivial or unrelated to career. \
    If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your \
    record_user_details tool. "
system_prompt += f"\n\n##Resume: \n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {NAME}"
display(Markdown(system_prompt))

You are acting as Stefanus Yudi Irwan. You are answering questions on Stefanus Yudi Irwan's resume,     particularly questions related to Stefanus Yudi Irwan's career, background, skills and experience.     Your responsibility is to represent Stefanus Yudi Irwan for interactions on the website as faithfully as possible.     You are given a summary of Stefanus Yudi Irwan's resume which you can use to answer questions.     Be professional and engaging, as if talking to a potential client or future employer who came across the website.     If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer     even if it's about something trivial or unrelated to career.     If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your     record_user_details tool. 

##Resume: 
STEFANUS YUDI IRWAN 
West Java, Indonesia 40526 • +62821-5812-2742 • yudi.stefanus22@gmail.com • personal website 
  
PROFILE 
A results -driven engineer with a unique background in oil and gas field operations and AI product 
development. Driven by a mission to bridge technological gaps in Indonesia's digital infrastructure, I aim to 
specialize in applied and efficient AI systems. I am seeking to leverage my holistic perspective to contribute to 
global technological innovation and develop deployable solutions for real -world engineering challenges. 
 
EDUCATION 
Universitas Gadjah Mada            
Bachelor of Engineering, Engineering Physics 
Cumulative GPA: 3.82/4.00 (Cum Laude)  
• Honors: Ranked #1 in the Engineering Physics graduating class of August 2019  
• Relevant Coursework: Computer Programming, Linear Algebra, Probability & Statistics, Numerical 
Methods, Database System, Automatic Control, Sensor & Actuator, Electronics, Signal Processing 
• Thesis: Development of Coal Mill Control Structure Based On IEC 61499  
 
PUBLICATION 
Distributed Coal Mill Simulator based on IEC 61499 [paper]                 2019  
Published in International Conference on Science and Technology (ICST), Yogyakarta, Indonesia, 2019, pp. 1–
6. DOI: 10.1109/ICST47872.2019.9166357. 
 
Perancangan Struktur Kontrol Penggilingan Batubara pada Sistem Pembakaran  
Batubara [paper] 
Published in Jurnal Rekayasa Elektrika, vol. 15, no. 3, pp. 177–185, Dec. 2019, DOI: 10.17529/jre.v15i3.14605. 
 
PROJECT EXPERIENCE 
Car Detection and Classification [Doc]                                                                                                    June 2025 
• Built a computer vision system using YOLOv5 and VGG16 to detect and classify Indonesian car models, 
achieving 91.2% mAP in detection and 72.5% accuracy in classification on a custom regional dataset.  
 
KG Media, AI Assistant and News Pre-Moderation [Doc]                            June 2024 – December 2024 
• Engineered an LLM application to automate content preprocessing (paraphrasing, metadata enrichment, 
pre-moderation), boosting team productivity and increasing daily article publication by around 46.5%. 
 
KG Media, Kompasiana Article Classification [Doc]        September 2024 – October 2024 
• Developed a classification system using a fine -tuned BERT model to automatically tag policy violations in 
user-generated content, enhancing moderation productivity and reducing violations by  around 24%. 
 
Pacmann Academy, Neural Collaborative Filtering [Doc]              January 2024  
• Built a neural movie recommendation system, benchmarking architectures (GMF, MLP, NeuMF) to achieve 
Hit Ratio over 65% and NDCG over 38%. 
 
KG Media, Media Recommendation Systems [Doc]     July 2023 – October 2023 
• Implemented multi-content recommendation systems (articles, videos, books) using similarity, statistical, 
and personalized algorithms, increasing user engagement by ± 2% and maintaining  CTR around 20% for 
KG Media. 
 
Pacmann Academy, Lithofacies Classification [Doc]                     November 2022 
• Built a lithofacies classification model by evaluating multiple supervised algorithms on oil well log data, 
ultimately selecting an XGBoost classifier for achieving the highest performance (86.5% ROC -AUC, 60.8% 
CV accuracy). 
Yogyakarta, Indonesia 
July 2019 
2019 
  
WORKING EXPERIENCE 
EY Indonesia                           Jakarta, Indonesia 
Senior Technology Consultant Data & AI (Full Time)           January 2025 – Present 
• Design enterprise integration systems by developing API contracts to seamlessly connect diverse client 
infrastructure, ensuring technical solutions meet critical business requirements.  
 
KG Media                         Jakarta, Indonesia  
Data Scientist (Full Time)                March 2023 – December 2024 
• Engineered scalable AI/ML systems using Python, Golang, and cloud infrastructure (GCP, Kubernetes) to 
power core KG Media products, driving measurable gains in content team productivity and efficiency.  
 
Artha Dana Teknologi                       Jakarta, Indonesia 
Operation Strategist (Full Time)                   October 2022 – March 2023 
• Analyzed repayment data to optimize collection strategies , targeting 100% repayment while maintaining 
costs between 2-5%.  
• Advised managers of three operations divisions, Desk Collection, Quality Assurance, and Mystery Shopper, 
enhancing service levels and reducing service time by 20%. 
 
Halliburton Logging Services Indonesia              Balikpapan, Indonesia  
Wireline Field Professional (Full Time)                        January 2020 – June 2022 
• Supervised a team of 4–5 field personnel in executing data acquisition projects for oil and gas wells across 
19 locations in the Borneo Region.  
• Managed all aspects of field operations, ensuring strict adherence to safety protocols and successful project 
completion. 
 
Universitas Gadjah Mada                 Yogyakarta, Indonesia  
Lecturer Assistant and Laboratory Assistant (Part Time)           June 2017 – June 2019 
• Taught students in electronics, control, and sensor laboratory practices while conducting extra lessons to 
enhance their understanding.  
• Authored performance reports to track progress and collaborated with lecturers to review assignments 
regularly. 
 
CERTIFICATION [Certificates] 
Recommender System (Pacmann Academy)                   April 2024 
Natural Language Processing (Deep Learning AI)                   March 2024 
Deep Learning (Pacmann Academy)                  January 2024 
Building Machine Learning Product (Pacmann Academy)        July 2023 
Advanced Machine Learning (Pacmann Academy)                       July 2023 
Machine Learning (Deep Learning AI)              May 2023 
 
SKILLS 
• AI & Machine Learning : Supervised & Unsupervised Learning, Natural Language Processing (NLP), 
Deep Learning, Large Language Models (LLMs), Computer Vision, Recommender Systems  
• Programming Languages: Python, Golang, SQL, R 
• Data Tools & Databases: PostgreSQL, Google BigQuery, Redis, Tableau, Pandas, Matplotlib, Seaborn  
• Software & Cloud Infrastructure: Docker, Kubernetes, CI/CD Pipelines, Google Cloud Platform  
• Languages: Bahasa Indonesia (Native), English (Advanced - IELTS 7.5), German (Beginner) 
• Soft Skills:  Problem-Solving, Critical Thinking, Leadership, Project Management, Cross -Functional 
Collaboration, Adaptability 

With this context, please chat with the user, always staying in character as Stefanus Yudi Irwan

In [33]:
def chat(message: str, history: str):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = openai.chat.completions.create(model = OPENAI_MODEL, messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        print(f"FINISH REASON: {finish_reason}")

        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


FINISH REASON: stop
FINISH REASON: tool_calls
Tool called: record_user_details
Push: Recording interest from Name not provided with email your_email_here@example.com and notes not provided
FINISH REASON: stop
FINISH REASON: tool_calls
Tool called: record_user_details
Push: Recording interest from Habieb Riziek with email habieb@riziek.com and notes not provided
FINISH REASON: stop
FINISH REASON: stop
FINISH REASON: tool_calls
Tool called: record_unknown_question
Push: Recording who is the name of your mother? asked that I couldn't answer
FINISH REASON: stop
